In [1]:
import psutil

# Get virtual memory details
mem_info = psutil.virtual_memory()

total_ram_gb = mem_info.total / 1e9
available_ram_gb = mem_info.available / 1e9

print(f"Total RAM limit: {total_ram_gb:.1f} GB")
print(f"Currently available RAM: {available_ram_gb:.1f} GB")

Total RAM limit: 54.8 GB
Currently available RAM: 53.1 GB


In [2]:
### Check for connected runtime (GPU or not)
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
### Check for more RAM
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 54.8 gigabytes of available RAM

You are using a high-RAM runtime!


In [4]:
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))

NameError: name 'torch' is not defined

##### Mount Drive into Collab

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##### Path to Model Repository

In [2]:
%cd /content/drive/MyDrive/Colab_Notebooks/AnySat

/content/drive/MyDrive/Colab_Notebooks/AnySat


##### Installing Model requirements

In [3]:
%pip install -r requirements.txt

# [ORIGINAL PART]

# AnySat Guide

#### Simple Usage

In [4]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import rasterio
from datetime import datetime
import h5py
import time
import psutil
import matplotlib.pyplot as plt
from IPython.display import clear_output

## AnySat is available through
### 1) PyTorch Hub
#### or
### 2) local repository

In [5]:
# 1) PyTorch Hub
#model = torch.hub.load('gastruc/anysat', 'anysat', pretrained=True, force_reload=True, flash_attn=False, trust_repo='check')

# 2) local repo
from hubconf import AnySat

model = AnySat.from_pretrained('base', flash_attn=False) #Set flash_attn=True if you have flash-attn module installed (url flash attn)
#device = "cuda" If you want to run on GPU default is cpu

Downloading: "https://huggingface.co/g-astruc/AnySat/resolve/main/models/AnySat.pth" to /root/.cache/torch/hub/checkpoints/AnySat.pth


100%|██████████| 480M/480M [00:20<00:00, 24.0MB/s]


### Added code to assure that every part of the Model runs on GPU

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the entire model and its submodules to the GPU
model = model.to(device)

# Iterate through all model parameters and move them to the GPU if not already there
for param in model.parameters():
    if param.device != device:
        param.data = param.data.to(device)

# Check for any remaining submodules not on the GPU
for name, module in model.named_modules():
    if next(module.parameters(), None) is not None and next(module.parameters()).device != device:  # Checks if module has parameters
        module.to(device)
        print(f"Module '{name}' moved to {device}")

# Feature Extraction

### Complete Folder

In [ ]:
import os
import torch
import glob
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

# 🔧 Configuration

feature_set = 'test' # either 'train' or 'test'
NUM_FILES = 2773      # either 8688(+1) for 'train' or 2772(+1) for 'test'
ORBIT = 'asc'        # either 'asc' or 'desc'
OUTPUT_TYPE = 'dense'  # Either 'tile', 'patch', 'dense'
CHANNELS = 3

BATCH_SIZE = 4        # ✅ Set your optimal batch size here
NUM_WORKERS = 8       # ✅ Set your optimal worker count here
PATCH_SIZE = 80       # ✅ Set your optimal patch size here (320 --> Out-of-Memory Error with A100 Extended RAM (89GB))

# 📁 Paths

if CHANNELS == 3:
  CHANNEL_FOLDER = "3_channels_VV_VH_ratio"
elif CHANNELS == 2:
  CHANNEL_FOLDER = "2_channels_VV_VH"
else:
  raise ValueError("Invalid number of channels. Must be 2 or 3.")


INPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/STANDARDIZED_{feature_set.upper()}_FEATURES_s1_{CHANNEL_FOLDER}_{ORBIT.lower()}_July"
OUTPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/EMBEDDED_{feature_set.upper()}_FEATURES_s1_{ORBIT}_July_PatchSize_{PATCH_SIZE}_OutputType_{OUTPUT_TYPE}"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
existing_outputs = set(os.listdir(OUTPUT_FOLDER))

model_used = model  # your pretrained model should already be defined

# 📦 Dataset
class TensorDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        tensor = torch.load(path).float()  # Shape: [# channels (2 or 3), 256, 256]
        filename = os.path.basename(path)
        return tensor, filename

# 📁 Load file paths
all_paths = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))[:NUM_FILES]
#all_paths = [sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))[-1]]

# Filter input tensor paths to exclude already-processed files
filtered_paths = []
for path in all_paths:
    original_name = os.path.basename(path)
    cleaned_name = original_name.removeprefix("normalized_")
    expected_output_name = f"Features_{ORBIT}_PatchSize_{PATCH_SIZE}_OutputType_{OUTPUT_TYPE}_{cleaned_name}"
    if expected_output_name not in existing_outputs:
        filtered_paths.append(path)

print(f"✅ {len(filtered_paths)} of {len(all_paths)} files will be processed (skipping existing ones)")

#dataset = TensorDataset(all_paths)
dataset = TensorDataset(filtered_paths)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# 📤 Inference & Save
with torch.no_grad():
    for batch, filenames in tqdm(loader, desc="Extracting features"):
        batch = batch.unsqueeze(1).to(device)  # [B, 1, # channels (2 or 3), 256, 256]
        dates = torch.tensor([196] * batch.shape[0]).to(device)

        data = {
            "s1": batch,
            "s1_dates": dates
        }

        features = model_used(data, patch_size=PATCH_SIZE, output=OUTPUT_TYPE, output_modality = 's1')  # Feature tensor output from your model

        # 💾 Save infered encoded features
        for i in range(len(filenames)):
            original_name = filenames[i]
            cleaned_name = original_name.removeprefix("normalized_")
            output_path = os.path.join(OUTPUT_FOLDER, f"Features_{ORBIT}_PatchSize_{PATCH_SIZE}_OutputType_{OUTPUT_TYPE}_{cleaned_name}")

            torch.save(features[i].cpu(), output_path)
            #print(f"✅ Saved {output_path}")



        # 🧹 Memory cleanup
        del batch, filenames, data, features
        torch.cuda.empty_cache()

print("✅ All encoded features extracted and saved.")

# PATCH_SIZE CHECK

In [ ]:
import os
import torch
import glob
import torch.nn.functional as F

# 🔧 Configuration
feature_set = 'test'
ORBIT = 'asc'
OUTPUT_TYPE = 'dense'
CHANNELS = 3

# ✅ Define the patch sizes you want to compare
# (Keeping sizes under 320 to avoid the OOM error you encountered)
PATCH_SIZES_TO_TEST = [80, 40, 160]

# 📁 Paths
CHANNEL_FOLDER = "3_channels_VV_VH_ratio" if CHANNELS == 3 else "2_channels_VV_VH"
INPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/{CHANNEL_FOLDER}/STANDARDIZED_{feature_set.upper()}_FEATURES_s1_{CHANNELS}channels_{ORBIT.lower()}_July"

# Ensure model is in evaluation mode for deterministic outputs
model_used = model
model_used.eval()

# 📁 Load a SINGLE file for the experiment
all_paths = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))
if not all_paths:
    raise FileNotFoundError(f"No files found in {INPUT_FOLDER}")

target_path = all_paths[0] # Selecting the first image in the folder
filename = os.path.basename(target_path)
print(f"🔬 Running patch_size experiment on: {filename}")

# Load and prepare the single tensor
# Original shape: [Channels, 256, 256] -> Need [Batch=1, Time=1, Channels, 256, 256]
single_tensor = torch.load(target_path).float().unsqueeze(0).unsqueeze(1).to(device)
dates = torch.tensor([196]).to(device)

data = {
    "s1": single_tensor,
    "s1_dates": dates
}

# Dictionary to store features for comparison
extracted_features = {}

# 📤 Inference Loop
with torch.no_grad():
    for p_size in PATCH_SIZES_TO_TEST:
        print(f"Extracting features with patch_size = {p_size}...")

        # Forward pass
        features = model_used(data, patch_size=p_size, output=OUTPUT_TYPE, output_modality='s1')

        # Squeeze out the batch dimension and move to CPU to save VRAM
        # Expected shape: [256, 256, 1536] (or similar depending on AnySat dense formatting)
        extracted_features[p_size] = features.squeeze(0).cpu()

        # 🧹 Memory cleanup per loop
        torch.cuda.empty_cache()

print("\n✅ Extraction complete. Proceeding to tensor comparison...\n")
print("-" * 50)

# 🔍 Comparison Step
# We will use the first patch size in our list as the baseline to compare against the others
base_size = PATCH_SIZES_TO_TEST[0]
base_tensor = extracted_features[base_size]

print(f"📏 Baseline Shape (patch_size={base_size}): {list(base_tensor.shape)}")
print("-" * 50)

for p_size in PATCH_SIZES_TO_TEST[1:]:
    compare_tensor = extracted_features[p_size]

    print(f"📊 Comparing patch_size {base_size} vs {p_size}:")

    # --- NEW: Shape Check ---
    shapes_match = base_tensor.shape == compare_tensor.shape
    print(f"   -> Shape Match: {shapes_match} (Compare shape: {list(compare_tensor.shape)})")

    if not shapes_match:
        print("   -> ⚠️ Shapes do not match! Skipping mathematical comparisons for this pair.")
        print("-" * 50)
        continue # Skips to the next patch_size in the loop to avoid a crash
    # ------------------------

    # 1. Exact Equality: Are every single float values identical?
    is_identical = torch.equal(base_tensor, compare_tensor)

    # 2. Mean Squared Error (MSE): How large is the average squared difference?
    mse = F.mse_loss(base_tensor, compare_tensor).item()

    # 3. Mean Absolute Error (MAE): Average absolute magnitude of differences
    mae = F.l1_loss(base_tensor, compare_tensor).item()

    print(f"   -> Exactly Identical: {is_identical}")
    if not is_identical:
        print(f"   -> Mean Squared Error (MSE): {mse:.8f}")
        print(f"   -> Mean Absolute Error (MAE): {mae:.8f}")

        # Check max deviation to see the worst-case difference
        max_diff = torch.max(torch.abs(base_tensor - compare_tensor)).item()
        print(f"   -> Max Absolute Difference:  {max_diff:.8f}")
    print("-" * 50)

🔬 Running patch_size experiment on: asc_3x256x256_00a28320_S1_10.pt
Extracting features with patch_size = 80...
Extracting features with patch_size = 40...
Extracting features with patch_size = 160...

✅ Extraction complete. Proceeding to tensor comparison...

--------------------------------------------------
📏 Baseline Shape (patch_size=80): [256, 256, 1536]
--------------------------------------------------
📊 Comparing patch_size 80 vs 40:
   -> Shape Match: True (Compare shape: [256, 256, 1536])
   -> Exactly Identical: False
   -> Mean Squared Error (MSE): 0.24028771
   -> Mean Absolute Error (MAE): 0.28969648
   -> Max Absolute Difference:  4.74420261
--------------------------------------------------
📊 Comparing patch_size 80 vs 160:
   -> Shape Match: True (Compare shape: [256, 256, 1536])
   -> Exactly Identical: False
   -> Mean Squared Error (MSE): 0.29640037
   -> Mean Absolute Error (MAE): 0.31671989
   -> Max Absolute Difference:  5.29992867
------------------------------

In [9]:
import os
import torch
import glob
import torch.nn.functional as F

# 🔧 Configuration
feature_set = 'test'
ORBIT = 'asc'
OUTPUT_TYPE = 'dense'
CHANNELS = 2

# ✅ Define the patch sizes you want to compare
# (Keeping sizes under 320 to avoid the OOM error you encountered)
PATCH_SIZES_TO_TEST = [80, 40, 160]

# 📁 Paths
CHANNEL_FOLDER = "3_channels_VV_VH_ratio" if CHANNELS == 3 else "2_channels_VV_VH"
INPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/{CHANNEL_FOLDER}/STANDARDIZED_{feature_set.upper()}_FEATURES_s1_{CHANNELS}channels_{ORBIT.lower()}_July"

# 📁 Dedicated folder for this experiment's outputs
EXPERIMENT_OUTPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/patch_size_experiment_{CHANNELS}channels"
os.makedirs(EXPERIMENT_OUTPUT_FOLDER, exist_ok=True)

model_used = model
model_used.eval()

# 📁 Load a SINGLE file for the experiment
all_paths = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))
if not all_paths:
    raise FileNotFoundError(f"No files found in {INPUT_FOLDER}")

target_path = all_paths[0]
filename = os.path.basename(target_path)
print(f"🔬 Running patch_size experiment on: {filename}")

single_tensor = torch.load(target_path).float().unsqueeze(0).unsqueeze(1).to(device)
dates = torch.tensor([196]).to(device)

data = {
    "s1-asc": single_tensor,
    "s1-asc_dates": dates
}

# Dictionary to store features and file metadata
extracted_features = {}

# 📤 Inference & Saving Loop
with torch.no_grad():
    for p_size in PATCH_SIZES_TO_TEST:
        print(f"Extracting & saving features with patch_size = {p_size}...")

        # Forward pass
        features = model_used(data, patch_size=p_size, output=OUTPUT_TYPE, output_modality='s1-asc')
        feature_tensor = features.squeeze(0).cpu()

        # Save to disk to check file size
        output_filename = f"Experiment_Features_Patch_{p_size}.pt"
        output_path = os.path.join(EXPERIMENT_OUTPUT_FOLDER, output_filename)
        torch.save(feature_tensor, output_path)

        # Get file size in Megabytes (MB)
        file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

        # Store data for the comparison step
        extracted_features[p_size] = {
            "tensor": feature_tensor,
            "size_mb": file_size_mb
        }

        # 🧹 Memory cleanup per loop
        torch.cuda.empty_cache()

print("\n✅ Extraction and Saving complete. Proceeding to tensor comparison...\n")
print("-" * 50)

# 🔍 Comparison Step
base_size = PATCH_SIZES_TO_TEST[0]
base_data = extracted_features[base_size]
base_tensor = base_data["tensor"]
base_file_size = base_data["size_mb"]

print(f"📏 Baseline (patch_size={base_size}):")
print(f"   -> Shape: {list(base_tensor.shape)}")
print(f"   -> File Size: {base_file_size:.2f} MB")
print("-" * 50)

for p_size in PATCH_SIZES_TO_TEST[1:]:
    compare_data = extracted_features[p_size]
    compare_tensor = compare_data["tensor"]
    compare_file_size = compare_data["size_mb"]

    print(f"📊 Comparing patch_size {base_size} vs {p_size}:")

    # --- Shape Check ---
    shapes_match = base_tensor.shape == compare_tensor.shape
    print(f"   -> Shape Match: {shapes_match} (Compare shape: {list(compare_tensor.shape)})")

    # --- File Size Check ---
    size_diff_mb = abs(base_file_size - compare_file_size)
    print(f"   -> File Size: {compare_file_size:.2f} MB (Difference: {size_diff_mb:.4f} MB)")

    if not shapes_match:
        print("   -> ⚠️ Shapes do not match! Skipping mathematical comparisons.")
        print("-" * 50)
        continue

    # Mathematical Checks
    is_identical = torch.equal(base_tensor, compare_tensor)
    mse = F.mse_loss(base_tensor, compare_tensor).item()
    mae = F.l1_loss(base_tensor, compare_tensor).item()

    print(f"   -> Exactly Identical: {is_identical}")
    if not is_identical:
        print(f"   -> Mean Squared Error (MSE): {mse:.8f}")
        print(f"   -> Mean Absolute Error (MAE): {mae:.8f}")
        max_diff = torch.max(torch.abs(base_tensor - compare_tensor)).item()
        print(f"   -> Max Absolute Difference:  {max_diff:.8f}")
    print("-" * 50)

🔬 Running patch_size experiment on: normalized_asc_2x256x256_00a28320_S1_10.pt
Extracting & saving features with patch_size = 80...
Extracting & saving features with patch_size = 40...
Extracting & saving features with patch_size = 160...

✅ Extraction and Saving complete. Proceeding to tensor comparison...

--------------------------------------------------
📏 Baseline (patch_size=80):
   -> Shape: [256, 256, 1536]
   -> File Size: 384.00 MB
--------------------------------------------------
📊 Comparing patch_size 80 vs 40:
   -> Shape Match: True (Compare shape: [256, 256, 1536])
   -> File Size: 384.00 MB (Difference: 0.0000 MB)
   -> Exactly Identical: False
   -> Mean Squared Error (MSE): 0.09524658
   -> Mean Absolute Error (MAE): 0.19664837
   -> Max Absolute Difference:  3.01802254
--------------------------------------------------
📊 Comparing patch_size 80 vs 160:
   -> Shape Match: True (Compare shape: [256, 256, 1536])
   -> File Size: 384.00 MB (Difference: 0.0000 MB)
   -> 

In [11]:
import os
import torch
import glob
import torch.nn.functional as F

# 🔧 Configuration
feature_set = 'test'
ORBIT = 'asc'
OUTPUT_TYPE = 'dense'
CHANNELS = 3

# ✅ Define the patch sizes you want to compare
# (Keeping sizes under 320 to avoid the OOM error you encountered)
PATCH_SIZES_TO_TEST = [80, 40, 160]

# 📁 Paths
CHANNEL_FOLDER = "3_channels_VV_VH_ratio" if CHANNELS == 3 else "2_channels_VV_VH"
INPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/{feature_set}_features/S1_pytorch_tensors_{feature_set}_features/{CHANNEL_FOLDER}/STANDARDIZED_{feature_set.upper()}_FEATURES_s1_{CHANNELS}channels_{ORBIT.lower()}_July"

# 📁 Dedicated folder for this experiment's outputs
EXPERIMENT_OUTPUT_FOLDER = f"/content/drive/MyDrive/Colab_Notebooks/Datasets/BioMassters/patch_size_experiment_{CHANNELS}channels"
os.makedirs(EXPERIMENT_OUTPUT_FOLDER, exist_ok=True)

model_used = model
model_used.eval()

# 📁 Load a SINGLE file for the experiment
all_paths = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))
if not all_paths:
    raise FileNotFoundError(f"No files found in {INPUT_FOLDER}")

target_path = all_paths[0]
filename = os.path.basename(target_path)
print(f"🔬 Running patch_size experiment on: {filename}")

single_tensor = torch.load(target_path).float().unsqueeze(0).unsqueeze(1).to(device)
dates = torch.tensor([196]).to(device)

data = {
    "s1": single_tensor,
    "s1_dates": dates
}

# Dictionary to store features and file metadata
extracted_features = {}

# 📤 Inference & Saving Loop
with torch.no_grad():
    for p_size in PATCH_SIZES_TO_TEST:
        print(f"Extracting & saving features with patch_size = {p_size}...")

        # Forward pass
        features = model_used(data, patch_size=p_size, output=OUTPUT_TYPE, output_modality='s1')
        feature_tensor = features.squeeze(0).cpu()

        # Save to disk to check file size
        output_filename = f"Experiment_Features_Patch_{p_size}.pt"
        output_path = os.path.join(EXPERIMENT_OUTPUT_FOLDER, output_filename)
        torch.save(feature_tensor, output_path)

        # Get file size in Megabytes (MB)
        file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

        # Store data for the comparison step
        extracted_features[p_size] = {
            "tensor": feature_tensor,
            "size_mb": file_size_mb
        }

        # 🧹 Memory cleanup per loop
        torch.cuda.empty_cache()

print("\n✅ Extraction and Saving complete. Proceeding to tensor comparison...\n")
print("-" * 50)

# 🔍 Comparison Step
base_size = PATCH_SIZES_TO_TEST[0]
base_data = extracted_features[base_size]
base_tensor = base_data["tensor"]
base_file_size = base_data["size_mb"]

print(f"📏 Baseline (patch_size={base_size}):")
print(f"   -> Shape: {list(base_tensor.shape)}")
print(f"   -> File Size: {base_file_size:.2f} MB")
print("-" * 50)

for p_size in PATCH_SIZES_TO_TEST[1:]:
    compare_data = extracted_features[p_size]
    compare_tensor = compare_data["tensor"]
    compare_file_size = compare_data["size_mb"]

    print(f"📊 Comparing patch_size {base_size} vs {p_size}:")

    # --- Shape Check ---
    shapes_match = base_tensor.shape == compare_tensor.shape
    print(f"   -> Shape Match: {shapes_match} (Compare shape: {list(compare_tensor.shape)})")

    # --- File Size Check ---
    size_diff_mb = abs(base_file_size - compare_file_size)
    print(f"   -> File Size: {compare_file_size:.2f} MB (Difference: {size_diff_mb:.4f} MB)")

    if not shapes_match:
        print("   -> ⚠️ Shapes do not match! Skipping mathematical comparisons.")
        print("-" * 50)
        continue

    # Mathematical Checks
    is_identical = torch.equal(base_tensor, compare_tensor)
    mse = F.mse_loss(base_tensor, compare_tensor).item()
    mae = F.l1_loss(base_tensor, compare_tensor).item()

    print(f"   -> Exactly Identical: {is_identical}")
    if not is_identical:
        print(f"   -> Mean Squared Error (MSE): {mse:.8f}")
        print(f"   -> Mean Absolute Error (MAE): {mae:.8f}")
        max_diff = torch.max(torch.abs(base_tensor - compare_tensor)).item()
        print(f"   -> Max Absolute Difference:  {max_diff:.8f}")
    print("-" * 50)

🔬 Running patch_size experiment on: asc_3x256x256_00a28320_S1_10.pt
Extracting & saving features with patch_size = 80...
Extracting & saving features with patch_size = 40...
Extracting & saving features with patch_size = 160...

✅ Extraction and Saving complete. Proceeding to tensor comparison...

--------------------------------------------------
📏 Baseline (patch_size=80):
   -> Shape: [256, 256, 1536]
   -> File Size: 384.00 MB
--------------------------------------------------
📊 Comparing patch_size 80 vs 40:
   -> Shape Match: True (Compare shape: [256, 256, 1536])
   -> File Size: 384.00 MB (Difference: 0.0000 MB)
   -> Exactly Identical: False
   -> Mean Squared Error (MSE): 0.24028771
   -> Mean Absolute Error (MAE): 0.28969648
   -> Max Absolute Difference:  4.74420261
--------------------------------------------------
📊 Comparing patch_size 80 vs 160:
   -> Shape Match: True (Compare shape: [256, 256, 1536])
   -> File Size: 384.00 MB (Difference: 0.0000 MB)
   -> Exactly Ide